# Preprocessing word embeddings and frequency lists
This notebook prepares the input data for the semantic network analysis. For each language, it aligns a frequency ranked unigram list with the corresponding word embedding file and extracts the 10,000 most frequent tokens for which valid 300-dimensional vectors are available.

The preprocessing includes:

- normalization of tokens to reduce Unicode inconsistencies,
- lowercasing and aggregation of frequency counts,
- filtering of invalid embedding lines,
- merging duplicate token variants using similarity weighted averaging,
- alignment of embeddings with the frequency list,
- saving the final top 10,000 word list and embedding matrix for each language.

The resulting `.npz` files contain the vocabulary and embedding matrix used in the cosine similarity and network construction steps.

In [ ]:
import os
import re
import unicodedata
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

## Token normalization

In [ ]:
def _canon_token(s: str) -> str:
    if not isinstance(s, str):
        return ""
    # Normalize to NFKC form (compatibility decomposition, then composition) to standardize characters.
    s = unicodedata.normalize("NFKC", s) 
    # Standardize apostrophe variants. 
    s =s.replace("’", "'").replace("‘", "'").replace("‛", "'").replace("′", "'")
    #Standardize hyphen and dash variants
    s = (s.replace("\u2010", "-").replace("\u2011", "-").replace("\u2012", "-").replace("\u2013", "-").replace("\u2014", "-"))
    #Normalize spaces and collapse repeated whitespace.
    s = s.replace("\u00A0", " ")
    s = " ".join(s.split())
    #Remove zero-width joiners. 
    s = s.replace("\u200c", "").replace("\u200d", "")
    return s.strip()

## Merging multiple token variants

Some embedding files contain several vectors that map to the same lowercase token after normalization. The function below merges duplicate token vectors into one 300dimensional representation. When more than two vectors are available, a similarity weighted average is used. Vectors that are more central among the duplicate variants receive higher weight.

In [ ]:
def weighted_averaging_embeddings(embeddings):
    """ Merge multiple vectors for the same token into one (300,) vector. 
    For a single vector, return it as is. For two  vectors, return the simple average. 
    For three or more vectors, compute the cosine similarity among them, and use the row-sum as centrality weights for a weighted average. 
    If the weights degenerate (e.g. all zero), use uniform weights instead. If no valid vectors, return a vector of NaNs.

    Returns:
        A single (300,) float32 vector representing the merged embedding for the token.

    """
    # Convert to numpy arrays and filter out invalid vectors (not 300-dimensional).
    E = []
    for e in embeddings:
        a = np.asarray(e, dtype=np.float32).reshape(-1)
        if a.size == 300:
            E.append(a)
    if not E:
        return np.full((300,), np.nan, dtype=np.float32) # no valid vectors, return NaNs
    
    # Stack valid vectors into a 2D array of shape (n, 300)
    E = np.stack(E, axis=0)     
    n = E.shape[0]
    if n == 1: # only one valid vector, return it as is
        return E[0]              
    if n == 2: # two vectors, return the simple average
        return E.mean(axis=0)    

    # Cosine similarity among duplicate token vectors
    S = cosine_similarity(E)     
    np.fill_diagonal(S, 0.0)    

    #Use row-sum of cosine similarity as centrality weights for weighted average.
    w = S.sum(axis=1)     

    # Fall back to uniform weights if the similarity weights are invalid.
    if not np.isfinite(w).all() or w.sum() == 0:
        w = np.ones(n, dtype=np.float32) / n
    else:
        w = (w / w.sum()).astype(np.float32)

    # Compute the weighted average of the vectors using the weights
    return (w[:, None] * E).sum(axis=0).astype(np.float32)

## Main preprocessing loop

The main loop processes each language folder separately. For each language, it reads the unigram frequency file and the corresponding embedding file, normalizes both vocabularies, aligns them by token form, and keeps the 10,000 most frequent words with valid embeddings.

For each language, the script saves:
- a compressed `.npz` file containing the word list and embedding matrix,
- a `.tsv` file containing the selected top 10,000 unigrams and their frequencies.

In [ ]:
root_dir = os.path.join("data")
summary = []

# Strategy for merging duplicate vectors after token normalization.
# Options: "first", "average", or "weighted average".
method = "weighted average"

# Repeat for each language folder sepearately
for folder_name in os.listdir(root_dir):
    folder_path = os.path.join(root_dir, folder_name)
    if not os.path.isdir(folder_path) or folder_name.startswith("~$"):
        continue

    # Extract the two-letter language code from folder names such as "de_*" or "en_*".
    m = re.match(r"^([a-z]{2})_", folder_name)
    if not m:
        print(f"Skipping folder: {folder_name}")
        continue
    lang_code = m.group(1)

    # Define the expected input files for the current language.
    freq_file = os.path.join(folder_path, f"dedup.{lang_code}.words.unigrams.tsv")
    vec_file  = os.path.join(folder_path, f"subs.{lang_code}.1e6.vec")

    if not os.path.exists(freq_file):
        print(f" missing freq file for {lang_code}: {freq_file}")
        continue
    if not os.path.exists(vec_file):
        print(f" missing vec file for {lang_code}: {vec_file}")
        continue

    # Read the frequency list. The first row is skipped because the file contains metadata/header information.
    freq_df = pd.read_csv(freq_file, sep="\t", skiprows=1, names=["unigram", "unigram_freq"])
    # Normalize unigrams: strip, lowercase, and canonicalize 
    freq_df["unigram"] = freq_df["unigram"].astype(str).str.strip().str.lower() 
    # Normalize token forms for later matching with the embedding vocabulary.
    freq_df["unigram"] = freq_df["unigram"].map(_canon_token)

    # Convert frequencies to numeric values and discard invalid rows.
    freq_df["unigram_freq"] = pd.to_numeric(freq_df["unigram_freq"], errors="coerce")
    freq_df = freq_df.dropna(subset=["unigram_freq"])
    freq_df["unigram_freq"] = freq_df["unigram_freq"].astype(int) 


    # Aggregate frequencies for duplicate unigrams (after normalization), and sort by frequency. 
    freq_df = (                                                     
        freq_df.groupby("unigram", as_index=False)["unigram_freq"]
               .sum()
               .sort_values("unigram_freq", ascending=False)
    )
    freq_df = freq_df[~freq_df["unigram"].isin(["", "nan", "NaN"])] # remove empty or invalid unigrams

    # Read the embedding file and keep only valid lines witha a token and 300 dimensions. 
    vecs = []
    with open(vec_file, encoding="utf-8") as f:
        next(f, None)  
        for line in f:
            if line.startswith("#"):
                continue

            parts = line.strip().split()

            # Expected format: token followed by 300 embedding dimensions.
            if len(parts) != 301:
                continue
            word = parts[0]

        
            if word.startswith("</") or "_" in word:
                continue
            try:
                # take exactly first 300 dims after the token
                embedding = [float(x) for x in parts[1: 301]]
                emb = np.array(embedding, dtype=np.float32)
                if emb.size == 300:
                    token_lc = _canon_token(word.lower())
                    vecs.append((token_lc, emb))  
            except ValueError:
                continue

    if not vecs:
        print(f" no usable vectors in {vec_file}")
        continue

    vec_df = pd.DataFrame(vecs, columns=["word", "embedding"]).dropna()

    # Merge multiple vectors that map to the same normalized lowercase token
    if method == "first": # keep the first vector occurrence for each token
        vec_df = vec_df.drop_duplicates(subset="word", keep="first")

    
    elif method == "average": # use an unweighted mean across duplicate vectors for the same token. 
        vec_df = (
            vec_df.groupby("word")["embedding"]
                  .apply(lambda g: np.mean(np.stack(list(g), axis=0), axis=0))
                  .reset_index()
        )

    else:
        # Use weighted averaging based on cosine similarity among duplicate vectors for the same token.
        vec_df = (
            vec_df.groupby("word")["embedding"]
                  .apply(lambda g: weighted_averaging_embeddings(list(g)))
                  .reset_index()
        )
        vec_df["embedding"] = vec_df["embedding"].apply(
            lambda v: (
                np.asarray(v, dtype=np.float32).reshape(300,)
                if isinstance(v, (list, np.ndarray)) and np.asarray(v).size == 300
                else np.nan
            )
        )
        vec_df = vec_df.dropna(subset=["embedding"])


    # Apply token normalization  once more before merging to ensure consistent matching
    freq_df["unigram"] = freq_df["unigram"].map(_canon_token)  
    vec_df["word"]     = vec_df["word"].map(_canon_token)     

    merged_df = freq_df.merge(vec_df, how="inner", left_on="unigram", right_on="word")

    # Select the 10,000 most frequent tokens with available embeddings
    merged_df = merged_df.sort_values("unigram_freq", ascending=False)

    if len(merged_df) < 10000:
        print(f" Not enough vectors for {lang_code}: {len(merged_df)} found")
        continue
    top10k = merged_df.head(10000).reset_index(drop=True)

    # Save the selected embeddings and vocabulary as a compressed NPZ file. 
    embedding_matrix = np.stack(top10k["embedding"].values, axis=0)
    word_list = top10k["unigram"].astype(str).to_numpy()

    npz_output_file = os.path.join(folder_path, f"{lang_code}.top10k.embeddings.simw.npz")
    np.savez_compressed(npz_output_file, vectors=embedding_matrix, words=word_list)
    print(f"saved {len(top10k)} vectors to {npz_output_file}")

    # Save them as TSV (word + frequency)
    output_file = os.path.join(folder_path, f"{lang_code}.top10k.unigrams.by_freq.tsv")
    top10k[["unigram", "unigram_freq"]].to_csv(output_file, sep="\t", index=False, header=True)

    summary.append({
        "language": lang_code,
        "rows_saved": len(top10k),
        "num_unique_words": top10k["unigram"].nunique(),
    })


summary_df = pd.DataFrame(summary)
summary_csv = os.path.join(root_dir, "vec_10k_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print(f" summary saved to {summary_csv}")
